# Duplicate checks

Explore duplicate candidates in the configured PostGIS database. The notebook uses `DATABASE_URL`, falling back to the project's local default, and limits displayed results to 100 rows. Set `SqlMagic.autolimit` to `0` to run without an automatic limit.

In [ ]:
from sqlalchemy import create_engine
from sqlalchemy.engine import make_url

from stac_dupes.config import database_url

url = make_url(database_url())
if url.drivername == "postgresql":
    url = url.set(drivername="postgresql+psycopg")
engine = create_engine(url)

%load_ext sql
%config SqlMagic.autolimit = 100
%config SqlMagic.displaylimit = 100
%sql engine

## Temporal and geometry overlap

Candidate pairs whose sensing intervals and geometries overlap.

In [ ]:
%%sql
SELECT
    a.id AS item_a_id,
    a.catalog_url AS item_a_catalog,
    a.stac_id AS item_a_stac_id,
    a.processing_baseline,
    a.product_type,
    b.id AS item_b_id,
    b.catalog_url AS item_b_catalog,
    b.stac_id AS item_b_stac_id
FROM items AS a
JOIN items AS b
    ON a.id < b.id
   AND a.processing_baseline IS NOT DISTINCT FROM b.processing_baseline
   AND a.product_type IS NOT DISTINCT FROM b.product_type
   AND tstzrange(a.sensing_start, a.sensing_end, '[]')
       && tstzrange(b.sensing_start, b.sensing_end, '[]')
   AND ST_Intersects(a.geometry, b.geometry);

## Exact geometry and sensing time

In [ ]:
%%sql
SELECT
    a.id AS item_a_id,
    b.id AS item_b_id,
    a.stac_id AS item_a_stac_id,
    b.stac_id AS item_b_stac_id,
    a.processing_baseline,
    a.product_type
FROM items AS a
JOIN items AS b
    ON a.id < b.id
   AND a.processing_baseline IS NOT DISTINCT FROM b.processing_baseline
   AND a.product_type IS NOT DISTINCT FROM b.product_type
   AND a.sensing_start IS NOT DISTINCT FROM b.sensing_start
   AND a.sensing_end IS NOT DISTINCT FROM b.sensing_end
   AND ST_Equals(a.geometry, b.geometry);

## Reused STAC IDs

STAC IDs reused by items from different catalog roots.

In [ ]:
%%sql
SELECT
    a.stac_id,
    a.processing_baseline,
    a.product_type,
    a.catalog_url AS catalog_a,
    b.catalog_url AS catalog_b
FROM items AS a
JOIN items AS b
    ON a.id < b.id
   AND a.stac_id = b.stac_id
   AND a.catalog_url <> b.catalog_url
   AND a.processing_baseline IS NOT DISTINCT FROM b.processing_baseline
   AND a.product_type IS NOT DISTINCT FROM b.product_type;

# Measure and filter overlap

Find candidate pairs with matching product type and processing baseline, then filter them by temporal and geometry overlap. Temporal and geometry percentages are measured relative to the shorter interval and smaller footprint, respectively. Geometry areas use the EPSG:6933 global equal-area projection.

In [ ]:
min_temporal_overlap_seconds = 1.0
min_temporal_overlap_percent = 90.0
min_geometry_overlap_percent = 90.0
result_limit = 100

In [ ]:
%%sql
WITH candidates AS (
    SELECT
        a.collection_id,
        a.id AS item_a_id,
        a.catalog_url AS item_a_catalog,
        a.stac_id AS item_a_stac_id,
        b.id AS item_b_id,
        b.catalog_url AS item_b_catalog,
        b.stac_id AS item_b_stac_id,
        a.processing_baseline,
        a.product_type,
        a.sensing_start AS item_a_sensing_start,
        a.sensing_end AS item_a_sensing_end,
        b.sensing_start AS item_b_sensing_start,
        b.sensing_end AS item_b_sensing_end,
        a.geometry AS item_a_geometry,
        b.geometry AS item_b_geometry
    FROM items AS a
    JOIN items AS b
        ON a.id < b.id
       AND a.processing_baseline IS NOT DISTINCT FROM b.processing_baseline
       AND a.product_type IS NOT DISTINCT FROM b.product_type
       AND tstzrange(a.sensing_start, a.sensing_end, '[]')
           && tstzrange(b.sensing_start, b.sensing_end, '[]')
       AND ST_Intersects(a.geometry, b.geometry)
    WHERE a.sensing_start IS NOT NULL
      AND a.sensing_end IS NOT NULL
      AND b.sensing_start IS NOT NULL
      AND b.sensing_end IS NOT NULL
),
temporal_metrics AS (
    SELECT
        candidates.*,
        LEAST(item_a_sensing_end, item_b_sensing_end)
            - GREATEST(item_a_sensing_start, item_b_sensing_start)
            AS temporal_overlap,
        LEAST(
            item_a_sensing_end - item_a_sensing_start,
            item_b_sensing_end - item_b_sensing_start
        ) AS shorter_duration
    FROM candidates
),
temporal_scores AS (
    SELECT
        temporal_metrics.*,
        CASE
            WHEN EXTRACT(EPOCH FROM shorter_duration) = 0 THEN
                CASE
                    WHEN item_a_sensing_start = item_b_sensing_start
                     AND item_a_sensing_end = item_b_sensing_end
                    THEN 100.0
                    ELSE 0.0
                END
            ELSE LEAST(
                100.0,
                GREATEST(
                    0.0,
                    100.0 * EXTRACT(EPOCH FROM temporal_overlap)
                        / EXTRACT(EPOCH FROM shorter_duration)
                )
            )
        END AS temporal_overlap_percent
    FROM temporal_metrics
),
projected_geometries AS MATERIALIZED (
    SELECT
        temporal_scores.*,
        ST_Transform(item_a_geometry, 6933) AS item_a_projected,
        ST_Transform(item_b_geometry, 6933) AS item_b_projected
    FROM temporal_scores
    WHERE EXTRACT(EPOCH FROM temporal_overlap)
              >= {{ min_temporal_overlap_seconds }}
      AND temporal_overlap_percent >= {{ min_temporal_overlap_percent }}
),
spatial_metrics AS MATERIALIZED (
    SELECT
        projected_geometries.*,
        ST_Equals(item_a_geometry, item_b_geometry) AS identical_geometry,
        ST_Area(
            ST_Intersection(item_a_projected, item_b_projected)
        ) AS overlap_sq_m,
        LEAST(
            ST_Area(item_a_projected),
            ST_Area(item_b_projected)
        ) AS smaller_geometry_sq_m
    FROM projected_geometries
),
scored AS (
    SELECT
        spatial_metrics.*,
        CASE
            WHEN identical_geometry THEN 100.0
            ELSE LEAST(
                100.0,
                GREATEST(
                    0.0,
                    100.0 * overlap_sq_m
                        / NULLIF(smaller_geometry_sq_m, 0)
                )
            )
        END AS geometry_overlap_percent
    FROM spatial_metrics
)
SELECT
    collection_id,
    item_a_id,
    item_a_catalog,
    item_a_stac_id,
    item_b_id,
    item_b_catalog,
    item_b_stac_id,
    processing_baseline,
    product_type,
    item_a_sensing_start,
    item_a_sensing_end,
    item_b_sensing_start,
    item_b_sensing_end,
    temporal_overlap,
    temporal_overlap_percent,
    identical_geometry,
    overlap_sq_m / 1000000.0 AS overlap_sq_km,
    geometry_overlap_percent
FROM scored
WHERE geometry_overlap_percent >= {{ min_geometry_overlap_percent }}
ORDER BY
    geometry_overlap_percent DESC,
    temporal_overlap_percent DESC,
    temporal_overlap DESC
LIMIT {{ result_limit }};